# CXR-Detect — Exploratory Data Analysis

This notebook is for understanding the dataset only. No training code lives here.

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml
from PIL import Image

sys.path.insert(0, '../src')

DATA_ROOT = Path('../data/chest_xray')
CLASSES   = ['NORMAL', 'PNEUMONIA']

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

random.seed(42)
print('Setup complete')

## 1. Dataset counts

In [ ]:
for split in ['train', 'val', 'test']:
    print(f'\n{split}/')
    for cls in CLASSES:
        folder = DATA_ROOT / split / cls
        n = len(list(folder.glob('*.jpeg')) + list(folder.glob('*.jpg')) + list(folder.glob('*.png')))
        print(f'  {cls}: {n}')

## 2. Sample images

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 7))

for row, cls in enumerate(CLASSES):
    folder = DATA_ROOT / 'train' / cls
    files  = random.sample(list(folder.iterdir()), 6)
    for col, f in enumerate(files):
        img = Image.open(f).convert('L')
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(cls, fontsize=8)
        axes[row, col].axis('off')

plt.suptitle('Sample images — Normal (top) vs Pneumonia (bottom)')
plt.tight_layout()
plt.show()

## 3. Class imbalance

In [ ]:
counts = {}
for cls in CLASSES:
    files = list((DATA_ROOT / 'train' / cls).iterdir())
    counts[cls] = len(files)

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#2980b9', '#c0392b']
bars = ax.bar(counts.keys(), counts.values(), color=colors)

for bar, (cls, n) in zip(bars, counts.items()):
    ax.text(bar.get_x() + bar.get_width()/2, n + 30, str(n), ha='center', fontsize=11)

ax.set_ylabel('Count')
ax.set_title('Class distribution — train split')
plt.tight_layout()
plt.show()

ratio = counts['PNEUMONIA'] / counts['NORMAL']
print(f'Imbalance ratio: 1:{ratio:.1f} (Normal:Pneumonia)')
print('This is why we use WeightedRandomSampler in dataset.py')

## 4. Image size distribution

In [ ]:
widths  = {cls: [] for cls in CLASSES}
heights = {cls: [] for cls in CLASSES}

for cls in CLASSES:
    folder = DATA_ROOT / 'train' / cls
    files  = random.sample(list(folder.iterdir()), 200)
    for f in files:
        w, h = Image.open(f).size
        widths[cls].append(w)
        heights[cls].append(h)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = ['#2980b9', '#c0392b']

for cls, c in zip(CLASSES, colors):
    axes[0].hist(widths[cls],  bins=30, alpha=0.7, label=cls, color=c)
    axes[1].hist(heights[cls], bins=30, alpha=0.7, label=cls, color=c)

axes[0].set_title('Width distribution');  axes[0].set_xlabel('Pixels'); axes[0].legend()
axes[1].set_title('Height distribution'); axes[1].set_xlabel('Pixels'); axes[1].legend()

plt.tight_layout()
plt.show()
print('Images vary widely in size — this is why we resize to 224x224 in get_transforms()')

## 5. Mean intensity per class

In [ ]:
means = {cls: [] for cls in CLASSES}
stds  = {cls: [] for cls in CLASSES}

for cls in CLASSES:
    folder = DATA_ROOT / 'train' / cls
    files  = random.sample(list(folder.iterdir()), 200)
    for f in files:
        arr = np.array(Image.open(f).convert('L'))
        means[cls].append(arr.mean())
        stds[cls].append(arr.std())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = ['#2980b9', '#c0392b']

for cls, c in zip(CLASSES, colors):
    axes[0].hist(means[cls], bins=30, alpha=0.7, label=cls, color=c)
    axes[1].hist(stds[cls],  bins=30, alpha=0.7, label=cls, color=c)

axes[0].set_title('Mean pixel intensity'); axes[0].set_xlabel('Intensity'); axes[0].legend()
axes[1].set_title('Pixel std (contrast)'); axes[1].set_xlabel('Std');       axes[1].legend()

plt.tight_layout()
plt.show()
print('Pneumonia X-rays tend to have higher mean intensity due to lung consolidation')

## 6. Average image per class

In [ ]:
SIZE = 224
avg  = {}

for cls in CLASSES:
    folder = DATA_ROOT / 'train' / cls
    files  = random.sample(list(folder.iterdir()), 100)
    arrs   = [np.array(Image.open(f).convert('L').resize((SIZE, SIZE))).astype(float) for f in files]
    avg[cls] = np.mean(arrs, axis=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].imshow(avg['NORMAL'],    cmap='gray'); axes[0].set_title('Avg Normal');    axes[0].axis('off')
axes[1].imshow(avg['PNEUMONIA'], cmap='gray'); axes[1].set_title('Avg Pneumonia'); axes[1].axis('off')

diff = avg['PNEUMONIA'] - avg['NORMAL']
im = axes[2].imshow(diff, cmap='RdBu_r'); axes[2].set_title('Difference (Pneumonia - Normal)'); axes[2].axis('off')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()
print('Red regions = brighter in pneumonia on average (lung opacities)')

## 7. Augmentation preview

In [ ]:
from dataset import get_transforms

train_tf = get_transforms(224, 'train', cfg)

folder   = DATA_ROOT / 'train' / 'PNEUMONIA'
img_path = random.choice(list(folder.iterdir()))
img_pil  = Image.open(img_path).convert('RGB')

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

axes[0, 0].imshow(img_pil)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i in range(1, 10):
    row, col = divmod(i, 5)
    aug    = train_tf(img_pil)
    img_np = aug.permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
    axes[row, col].imshow(img_np)
    axes[row, col].set_title(f'Augmented {i}')
    axes[row, col].axis('off')

plt.suptitle('Same image — 9 different augmented versions')
plt.tight_layout()
plt.show()

## 8. Class weights check

In [ ]:
from dataset import XRayDataset, get_transforms

train_ds = XRayDataset(DATA_ROOT / 'train', get_transforms(224, 'train', cfg))
counts   = train_ds.class_counts()
total    = sum(counts)
weights  = [total / (len(counts) * c) for c in counts]

for cls, count, weight in zip(CLASSES, counts, weights):
    print(f'{cls}: {count} images  →  sampler weight = {weight:.3f}')

print(f'\nThe minority class (NORMAL) gets {weights[0]/weights[1]:.1f}x higher sampling weight')
print('This ensures balanced batches during training')